# Qwen 2.5 modelis

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True) #langsmithui
import feedparser
from langsmith import Client
from typing import Union
import json
import requests
from bs4 import BeautifulSoup
from datetime import date
from langgraph.prebuilt import create_react_agent
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field
from typing import List
import logging
import os
import warnings
import sys
import io
from langchain_core.globals import set_debug
import os.path
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError 

c:\Users\arnas\Desktop\Kursinis\venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
LLM_NAME = "qwen2.5" # modelio pavadinimas kuri naudosime
model = ChatOllama(model=LLM_NAME, temperature=0).bind(tool_choice= "required")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2") # semantines paieskos modelis
stderr = sys.stderr
sys.stderr = stderr
sys.stderr = io.StringIO()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
class Rungtynes(BaseModel):
    komanda1: str = Field(description="First team name")
    komanda2: str = Field(description="Second team name")
    data: str = Field(description="Match date in YYYY-MM-DD format")
    laikas: str | None = Field(default=None, description="Match time in HH:MM format, if available")

class RungtyniuSarasas(BaseModel):
    rungtynes: List[Rungtynes]

In [4]:
kategorijos = ["football", "nba", "euroleague", "lkl", "basketball_champions_league"]

In [5]:
SCOPES = ["https://www.googleapis.com/auth/calendar"]
set_debug(True) # ziureti galvojimo procesa agento
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"  
os.environ["HF_HUB_VERBOSITY"] = "error"
warnings.filterwarnings("ignore")                 
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR) # cia tiesiog paslepiame nereikalinga info kaip loading weights ar erorus


def leidimas_naudotis_kaledoriumi(): # suteikiam leidima kodui naudotis google kalendoriumi, pirma karta paleidus issoks langas kuriame reikes prisijungti
    creds = None
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPES)
            creds = flow.run_local_server(port=0)
        with open("token.json", "w") as token:
            token.write(creds.to_json())
    return build("calendar", "v3", credentials=creds)

In [6]:
@tool
def gauk_straipsniu_sarasa(kategorija: str, uzklausa: str) -> str:
    """ALWAYS call this tool FIRST. Finds relevant sports news articles by user query.
    Args:
        kategorija: sports category – one of: 'football', 'nba', 'euroleague', 'lkl', 'basketball_champions_league'.
        uzklausa: user query.
    """
    aplankas = kategorija.lower()
    if aplankas not in kategorijos:
        return f"Unknown category: '{kategorija}'. Available categories: {kategorijos}"

    docs = []
    for failas in os.listdir(aplankas):
        if failas.endswith(".json"):
            with open(f"{aplankas}/{failas}", "r", encoding="utf-8") as f:
                duomenys = json.load(f)
                pavadinimas = duomenys["pavadinimas"]
                aprasymas = duomenys["aprasymas"]
                failo_kelias = f"{aplankas}/{failas}"

                docs.append(Document(
                    page_content=f"{pavadinimas}. {aprasymas}",
                    metadata={"failas": failo_kelias}
                ))

    if not docs:
        return "No articles found for this category."

    db = FAISS.from_documents(docs, embeddings)
    rezultatai = db.similarity_search(uzklausa, k=5)

    atrinkti = [
        {"pavadinimas": r.page_content, "failas": r.metadata["failas"]}
        for r in rezultatai
    ]
    
    failai = [r.metadata["failas"] for r in rezultatai]
    
    return json.dumps({
        "straipsniai": atrinkti,
        "failai": failai
    }, ensure_ascii=False)


@tool
def gauk_straipsnio_teksta(failai: list[str], uzklausa: str) -> str:
    """Fetches full article text and returns the most relevant chunks by user query.
    Call this tool AFTER gauk_straipsniu_sarasa.
    ALWAYS pass ALL paths from 'failai' field received from gauk_straipsniu_sarasa, not just one.
    Args:
        failai: list of ALL file paths from gauk_straipsniu_sarasa 'failai' field.
        uzklausa: user query.
    """
    docs = []
    for failas in failai:
        try:
            with open(failas, "r", encoding="utf-8") as f:
                duomenys = json.load(f)
                tekstas = duomenys["straipsnis"]
                if tekstas:
                    docs.append(Document(
                        page_content=tekstas,
                        metadata={"failas": failas}
                    ))
        except Exception as e:
            print(f"Klaida: {failas}: {e}")

    # patikrinimas kad docs nera tuscias
    if not docs:
        return "No article content found for the provided file paths."

    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(docs)

    if not chunks:
        return "No content chunks generated."

    db = FAISS.from_documents(chunks, embeddings)
    rezultatai = db.similarity_search(uzklausa, k=4)

    aktualus = [{"tekstas": r.page_content} for r in rezultatai]
    return json.dumps(aktualus, ensure_ascii=False)


@tool
def isskirk_rungtynes(straipsniai_json: Union[str, list]) -> str:
    """Extracts upcoming match information (teams, date, time) from article text.
    Call this tool ALWAYS when user asks about upcoming matches or schedule.
    NEVER skip this tool if user asks about upcoming matches.
    Call this tool AFTER gauk_straipsnio_teksta.
    Args:
        straipsniai_json: article text from gauk_straipsnio_teksta results.
    """
    if isinstance(straipsniai_json, list): # buvo kad sarasu paduodavo tai konvertuojam i stringa
        straipsniai_json = json.dumps(straipsniai_json, ensure_ascii=False)

    llm = ChatOllama(model=LLM_NAME, temperature=0).with_structured_output(RungtyniuSarasas)
    rezultatas = llm.invoke(f"""
        Extract ONLY upcoming matches found in the text below.
        If no match information is found in the text - return empty list.
        Do NOT use any prior knowledge or training data.
        Only use information explicitly mentioned in the provided text.
        Convert dates to YYYY-MM-DD format.
        Convert time to HH:MM format.
        If date or time is not mentioned - leave empty.
        
        Text: {straipsniai_json}
    """)
    return rezultatas.model_dump_json()

@tool
def ideti_i_kalendoriu(rungtynes_json: Union[str, list, dict]) -> str:
    """Adds upcoming matches to Google Calendar.
    Call this tool AFTER isskirk_rungtynes, OR directly if match information is already in context.
    ALWAYS pass rungtynes_json as a JSON string in this exact format:
    '{"rungtynes": [{"komanda1": "Vilniaus Rytas", "komanda2": "Neptūnas", "data": "2026-06-01", "laikas": "19:00"}]}'
    Args:
        rungtynes_json: JSON string with match information.
    """
    if isinstance(rungtynes_json, (list, dict)):
        rungtynes_json = json.dumps(rungtynes_json, ensure_ascii=False)

    try:
        service = leidimas_naudotis_kaledoriumi()
        duomenys = json.loads(rungtynes_json)
        
        if "rungtynes" in duomenys:
            rungtynes = duomenys["rungtynes"]
        elif isinstance(duomenys, list):
            rungtynes = duomenys
        else:
            rungtynes = [duomenys]
        
        prideta = []
        for r in rungtynes:
            komanda1 = r.get("komanda1")
            komanda2 = r.get("komanda2")
            
            if not komanda1 or not komanda2:
                continue

            laikas = r.get("laikas")
            data = r.get("data")

            if laikas:
                pradzia = f"{data}T{laikas}:00"
                event = {
                    "summary": f"{komanda1} vs {komanda2}",
                    "start": {"dateTime": pradzia, "timeZone": "Europe/Vilnius"},
                    "end":   {"dateTime": pradzia, "timeZone": "Europe/Vilnius"},
                }
            else:
                event = {
                    "summary": f"{komanda1} vs {komanda2}",
                    "start": {"date": data},
                    "end":   {"date": data},
                }
            
            service.events().insert(calendarId="primary", body=event).execute()
            prideta.append(f"{komanda1} vs {komanda2} - {data}")
        
        return f"Successfully added to calendar: {', '.join(prideta)}"
    
    except Exception as e:
        return f"Error adding to calendar: {str(e)}"

In [7]:
siandiena = "2026-05-10"
leidimas_naudotis_kaledoriumi()
kontekstas = ""
llm_santraukai = ChatOllama(model=LLM_NAME, temperature=0)

def klausk(uzklausa: str, kontekstas: str) -> tuple[str, str]:
    agent = create_react_agent(
        model=model,
        tools=[gauk_straipsniu_sarasa, gauk_straipsnio_teksta, isskirk_rungtynes, ideti_i_kalendoriu],
        prompt=f"""
You are a Lithuanian sports news assistant. Today's date: {siandiena}.

{f"Papildomas kontekstas: {kontekstas}" if kontekstas else ""}

Always follow this EXACT order, never skip steps:
1. Call gauk_straipsniu_sarasa to find relevant articles.
2. ALWAYS call gauk_straipsnio_teksta with the file paths from step 1 before doing anything else.
3. If user asks about upcoming matches - call isskirk_rungtynes with the text from step 2.
4. If user asks to add matches to calendar:
   - If match information is already in "Papildomas kontekstas" - call ideti_i_kalendoriu directly with that information, skip steps 1-3.
   - Otherwise follow steps 1-3 first, then call ideti_i_kalendoriu.
5. Provide a summary in Lithuanian as a single continuous paragraph without any line breaks, bullet points or numbered lists.

When calling ideti_i_kalendoriu, ALWAYS pass rungtynes_json as a JSON string as in the example:
'{{"rungtynes": [{{"komanda1": "Vilniaus Rytas", "komanda2": "Neptūnas", "data": "2026-06-01", "laikas": "19:00"}}]}}'
""",
    )

    atsakymas = agent.invoke(
    {"messages": [{"role": "user", "content": uzklausa}]},
    config={
        "run_name": f"{LLM_NAME} | {uzklausa}",
        "tags": [LLM_NAME, "kursinis"],
        "metadata": {"modelis": LLM_NAME}
    }
 )

    tekstas = atsakymas["messages"][-1].content
    tekstas = " ".join(tekstas.split())

    kontekstas = llm_santraukai.invoke(f"""
    Extract key facts in Lithuanian (max 100 words).
    If matches were found, include them in this exact format:
    RUNGTYNĖS: {{"rungtynes": [{{"komanda1": "Team1", "komanda2": "Team2", "data": "YYYY-MM-DD", "laikas": "HH:MM or null"}}]}}
    
    User asked: {uzklausa}
    Agent answered: {tekstas}
    Previous context: {kontekstas}
    """).content

    if len(kontekstas.split()) > 100:
        kontekstas = llm_santraukai.invoke(f"""
            Shorten this text to 100 words while keeping the key facts:
            {kontekstas}
        """).content

    return tekstas, kontekstas

In [8]:
atsakymas1, kontekstas = klausk("Kokios artėjančios Žalgirio LKL rungtynės?", kontekstas)
print(atsakymas1)
print(f"Kontekstas: {kontekstas}")

[chain/start] [chain:qwen2.5 | Kokios artėjančios Žalgirio LKL rungtynės?] Entering Chain run with input:
{
  "messages": [
    {
      "role": "user",
      "content": "Kokios artėjančios Žalgirio LKL rungtynės?"
    }
  ]
}
[chain/start] [chain:qwen2.5 | Kokios artėjančios Žalgirio LKL rungtynės? > chain:agent] Entering Chain run with input:
[inputs]
[chain/start] [chain:qwen2.5 | Kokios artėjančios Žalgirio LKL rungtynės? > chain:agent > chain:call_model] Entering Chain run with input:
[inputs]
[chain/start] [chain:qwen2.5 | Kokios artėjančios Žalgirio LKL rungtynės? > chain:agent > chain:call_model > chain:RunnableSequence] Entering Chain run with input:
[inputs]
[chain/start] [chain:qwen2.5 | Kokios artėjančios Žalgirio LKL rungtynės? > chain:agent > chain:call_model > chain:RunnableSequence > chain:Prompt] Entering Chain run with input:
[inputs]
[chain/end] [chain:qwen2.5 | Kokios artėjančios Žalgirio LKL rungtynės? > chain:agent > chain:call_model > chain:RunnableSequence > chai

In [9]:
kontekstas = 'Žalgiris LKL artėjančios rungtynės. RUNGTYNĖS: {"rungtynes": [{"komanda1": "Kauno Žalgiris", "komanda2": "Utenos Juventus", "data": "2026-05-20", "laikas": "19:00"}]} '
atsakymas2, kontekstas = klausk("Įdėk rungtynes į kalendorių", kontekstas)
print(atsakymas2)
print(f"Santrauka: {kontekstas}")

[chain/start] [chain:qwen2.5 | Įdėk rungtynes į kalendorių] Entering Chain run with input:
{
  "messages": [
    {
      "role": "user",
      "content": "Įdėk rungtynes į kalendorių"
    }
  ]
}
[chain/start] [chain:qwen2.5 | Įdėk rungtynes į kalendorių > chain:agent] Entering Chain run with input:
[inputs]
[chain/start] [chain:qwen2.5 | Įdėk rungtynes į kalendorių > chain:agent > chain:call_model] Entering Chain run with input:
[inputs]
[chain/start] [chain:qwen2.5 | Įdėk rungtynes į kalendorių > chain:agent > chain:call_model > chain:RunnableSequence] Entering Chain run with input:
[inputs]
[chain/start] [chain:qwen2.5 | Įdėk rungtynes į kalendorių > chain:agent > chain:call_model > chain:RunnableSequence > chain:Prompt] Entering Chain run with input:
[inputs]
[chain/end] [chain:qwen2.5 | Įdėk rungtynes į kalendorių > chain:agent > chain:call_model > chain:RunnableSequence > chain:Prompt] s] Exiting Chain run with output:
[outputs]
[llm/start] [chain:qwen2.5 | Įdėk rungtynes į kalen

# Qwen3 modelis

In [10]:
LLM_NAME = "qwen3:8b"
model = ChatOllama(model=LLM_NAME, temperature=0).bind(tool_choice="required")
llm_santraukai = ChatOllama(model=LLM_NAME, temperature=0)
kontekstas = ""

In [11]:
atsakymas1, kontekstas = klausk("Kokios artėjančios Žalgirio LKL rungtynės?", kontekstas)
print(atsakymas1)
print(f"Kontekstas: {kontekstas}")

[chain/start] [chain:qwen3:8b | Kokios artėjančios Žalgirio LKL rungtynės?] Entering Chain run with input:
{
  "messages": [
    {
      "role": "user",
      "content": "Kokios artėjančios Žalgirio LKL rungtynės?"
    }
  ]
}
[chain/start] [chain:qwen3:8b | Kokios artėjančios Žalgirio LKL rungtynės? > chain:agent] Entering Chain run with input:
[inputs]
[chain/start] [chain:qwen3:8b | Kokios artėjančios Žalgirio LKL rungtynės? > chain:agent > chain:call_model] Entering Chain run with input:
[inputs]
[chain/start] [chain:qwen3:8b | Kokios artėjančios Žalgirio LKL rungtynės? > chain:agent > chain:call_model > chain:RunnableSequence] Entering Chain run with input:
[inputs]
[chain/start] [chain:qwen3:8b | Kokios artėjančios Žalgirio LKL rungtynės? > chain:agent > chain:call_model > chain:RunnableSequence > chain:Prompt] Entering Chain run with input:
[inputs]
[chain/end] [chain:qwen3:8b | Kokios artėjančios Žalgirio LKL rungtynės? > chain:agent > chain:call_model > chain:RunnableSequence 

In [12]:
kontekstas = 'Žalgiris LKL artėjančios rungtynės. RUNGTYNĖS: {"rungtynes": [{"komanda1": "Kauno Žalgiris", "komanda2": "Utenos Juventus", "data": "2026-05-20", "laikas": "19:00"}]} '
atsakymas2, kontekstas = klausk("Įdėk rungtynes į kalendorių", kontekstas)
print(atsakymas2)
print(f"Santrauka: {kontekstas}")

[chain/start] [chain:qwen3:8b | Įdėk rungtynes į kalendorių] Entering Chain run with input:
{
  "messages": [
    {
      "role": "user",
      "content": "Įdėk rungtynes į kalendorių"
    }
  ]
}
[chain/start] [chain:qwen3:8b | Įdėk rungtynes į kalendorių > chain:agent] Entering Chain run with input:
[inputs]
[chain/start] [chain:qwen3:8b | Įdėk rungtynes į kalendorių > chain:agent > chain:call_model] Entering Chain run with input:
[inputs]
[chain/start] [chain:qwen3:8b | Įdėk rungtynes į kalendorių > chain:agent > chain:call_model > chain:RunnableSequence] Entering Chain run with input:
[inputs]
[chain/start] [chain:qwen3:8b | Įdėk rungtynes į kalendorių > chain:agent > chain:call_model > chain:RunnableSequence > chain:Prompt] Entering Chain run with input:
[inputs]
[chain/end] [chain:qwen3:8b | Įdėk rungtynes į kalendorių > chain:agent > chain:call_model > chain:RunnableSequence > chain:Prompt] s] Exiting Chain run with output:
[outputs]
[llm/start] [chain:qwen3:8b | Įdėk rungtynes 

# Llama3.1 modelis

In [13]:
LLM_NAME = "llama3.1:8b"
model = ChatOllama(model=LLM_NAME, temperature=0).bind(tool_choice="required")
llm_santraukai = ChatOllama(model=LLM_NAME, temperature=0)
kontekstas = ""

In [14]:
atsakymas1, kontekstas = klausk("Kokios artėjančios Žalgirio LKL rungtynės?", kontekstas)
print(atsakymas1)
print(f"Kontekstas: {kontekstas}")

[chain/start] [chain:llama3.1:8b | Kokios artėjančios Žalgirio LKL rungtynės?] Entering Chain run with input:
{
  "messages": [
    {
      "role": "user",
      "content": "Kokios artėjančios Žalgirio LKL rungtynės?"
    }
  ]
}
[chain/start] [chain:llama3.1:8b | Kokios artėjančios Žalgirio LKL rungtynės? > chain:agent] Entering Chain run with input:
[inputs]
[chain/start] [chain:llama3.1:8b | Kokios artėjančios Žalgirio LKL rungtynės? > chain:agent > chain:call_model] Entering Chain run with input:
[inputs]
[chain/start] [chain:llama3.1:8b | Kokios artėjančios Žalgirio LKL rungtynės? > chain:agent > chain:call_model > chain:RunnableSequence] Entering Chain run with input:
[inputs]
[chain/start] [chain:llama3.1:8b | Kokios artėjančios Žalgirio LKL rungtynės? > chain:agent > chain:call_model > chain:RunnableSequence > chain:Prompt] Entering Chain run with input:
[inputs]
[chain/end] [chain:llama3.1:8b | Kokios artėjančios Žalgirio LKL rungtynės? > chain:agent > chain:call_model > chain

In [15]:
kontekstas = 'Žalgiris LKL artėjančios rungtynės. RUNGTYNĖS: {"rungtynes": [{"komanda1": "Kauno Žalgiris", "komanda2": "Utenos Juventus", "data": "2026-05-20", "laikas": "19:00"}]} '
atsakymas2, kontekstas = klausk("Įdėk rungtynes į kalendorių", kontekstas)
print(atsakymas2)
print(f"Santrauka: {kontekstas}")

[chain/start] [chain:llama3.1:8b | Įdėk rungtynes į kalendorių] Entering Chain run with input:
{
  "messages": [
    {
      "role": "user",
      "content": "Įdėk rungtynes į kalendorių"
    }
  ]
}
[chain/start] [chain:llama3.1:8b | Įdėk rungtynes į kalendorių > chain:agent] Entering Chain run with input:
[inputs]
[chain/start] [chain:llama3.1:8b | Įdėk rungtynes į kalendorių > chain:agent > chain:call_model] Entering Chain run with input:
[inputs]
[chain/start] [chain:llama3.1:8b | Įdėk rungtynes į kalendorių > chain:agent > chain:call_model > chain:RunnableSequence] Entering Chain run with input:
[inputs]
[chain/start] [chain:llama3.1:8b | Įdėk rungtynes į kalendorių > chain:agent > chain:call_model > chain:RunnableSequence > chain:Prompt] Entering Chain run with input:
[inputs]
[chain/end] [chain:llama3.1:8b | Įdėk rungtynes į kalendorių > chain:agent > chain:call_model > chain:RunnableSequence > chain:Prompt] s] Exiting Chain run with output:
[outputs]
[llm/start] [chain:llama3.1